# Particiones y Variables Comunes

**Proyecto 2. Resultados | LMSYS Chatbot Arena | CC3084 Data Science**

Este notebook genera **una sola vez** la partición train/val/test que usan los cuatro modelos, y
verifica las variables comunes. Si cada integrante partiera los datos por su cuenta, las métricas no
serían comparables. El resultado es `data/splits.csv` (columnas `id`, `split`), que se versiona en Git.

---
## 1. Configuración del Ambiente

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import load, splits, features

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

FIGURAS = ROOT / 'docs' / 'figures'
FIGURAS.mkdir(parents=True, exist_ok=True)

---
## 2. Carga del Dataset Limpio

Se parte de `data/processed/train_clean.csv`, generado por el notebook 02.

In [ ]:
df = load.load_train_clean()
print(f'{len(df):,} filas x {df.shape[1]} columnas')
print('Clases:', df['winner'].map(cfg.LABEL_NAMES).value_counts(normalize=True).round(4).to_dict())

---
## 3. Partición 70 / 15 / 15

Criterios:

- **Estratificada por `winner`**, para que las tres clases (incluido el empate) mantengan su proporción en cada partición.
- **Agrupada por el texto del prompt**: un mismo prompt, que puede aparecer en varias batallas con distintas parejas de modelos, queda siempre en una sola partición. Si no, el modelo vería en entrenamiento el mismo prompt que luego se evalúa en prueba.
- **Semilla fija (42)**, así que regenerarla da exactamente el mismo resultado.

Se usa `StratifiedGroupKFold` con 20 folds (≈ 5 % cada uno): 14 para entrenamiento, 3 para validación y 3 para prueba.

El conjunto de **prueba** solo se usa al final para reportar resultados. Los hiperparámetros se ajustan con validación.

In [ ]:
particion = splits.make_splits(df)
resumen = splits.check_splits(df, particion)
resumen

Las verificaciones de `check_splits` pasaron: todas las filas tienen partición y ningún prompt está repartido entre particiones. La tabla muestra que el tamaño y la proporción de clases son parecidos en las tres.

In [ ]:
m = df.merge(particion, on=cfg.COL_ID)
m['clase'] = m['winner'].map(cfg.LABEL_NAMES)
prop = (m.groupby('split')['clase'].value_counts(normalize=True).mul(100)
          .rename('porcentaje').reset_index())

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=prop, x='split', y='porcentaje', hue='clase', order=['train', 'val', 'test'], ax=ax)
ax.set_title('Distribución de clases por partición')
ax.set_xlabel('')
ax.set_ylabel('% de la partición')
sns.despine()
plt.tight_layout()
plt.savefig(FIGURAS / 'fig_particiones_clases.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Guardado de la Partición

`data/splits.csv` **sí se versiona en Git** (a diferencia de `data/processed/`, que está en el `.gitignore`).

In [ ]:
particion.to_csv(cfg.F_SPLITS, index=False)
print(f'Guardado en: {cfg.F_SPLITS}')

# Se vuelve a leer para comprobar que lo guardado es idéntico y reproducible
leida = load.load_splits()
assert leida.equals(particion)
assert splits.make_splits(df).equals(particion), 'La partición no es reproducible'
print('Reproducible: regenerarla con la misma semilla da el mismo resultado.')

---
## 5. Cómo Cargar la Partición en los Modelos

```python
from src import load, features

train = load.load_split('train')      # tambien 'val' y 'test'
X_train = features.build_base_features(train)
y_train = train['winner']
```

`load_split` devuelve el dataset limpio filtrado a esa partición, con `winner` (0 = gana A, 1 = gana B, 2 = empate).

In [ ]:
train = load.load_split('train')
val = load.load_split('val')
test = load.load_split('test')
print(f'train: {len(train):,} | val: {len(val):,} | test: {len(test):,}')

---
## 6. Variables Comunes

`features.build_base_features` construye las variables numéricas base (longitudes, palabras, turnos), sus
logaritmos `log(1 + x)`, las diferencias A − B y las dos categóricas (`model_a`, `model_b`). No incluye
`winner` ni `winner_*`: usarlos como entrada sería fuga de información.

In [ ]:
X = features.build_base_features(train)
prohibidas = {'winner', cfg.COL_WIN_A, cfg.COL_WIN_B, cfg.COL_WIN_TIE}
assert not prohibidas & set(X.columns), 'Hay columnas de etiqueta entre las variables'
print(f'{X.shape[1] - 1} variables (sin contar id)')
X.head()

---
## 7. Intercambio A/B

`features.swap_ab` intercambia las posiciones A↔B (modelos, textos, longitudes y etiqueta). Se usa para la
prueba de sesgo de posición, y también como aumento de datos. Se comprueba aquí que es una involución
(aplicarlo dos veces devuelve el original) y que no toca los empates.

In [ ]:
s = features.swap_ab(val)
assert features.swap_ab(s).equals(val), 'swap_ab no es una involución'
assert (s.loc[val['winner'] == 2, 'winner'] == 2).all()
assert (s['winner'] != val['winner'])[val['winner'] != 2].all()
print('swap_ab correcto.')

---
## 8. Resumen

| Elemento | Valor |
|---|---|
| Partición | 70 / 15 / 15, estratificada por clase y agrupada por prompt |
| Semilla | 42 |
| Archivo | `data/splits.csv` (`id`, `split`), versionado en Git |
| Carga | `load.load_split('train' \| 'val' \| 'test')` |
| Variables comunes | `features.build_base_features` |
| Prueba de posición | `features.swap_ab` |